In [80]:
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Annotated
from dotenv import load_dotenv
import operator

In [81]:
load_dotenv()

# Model
model = ChatGoogleGenerativeAI(model='gemini-3-flash-preview')

In [82]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description='Detailed feedback for the essay')
    score: int = Field(description='Score out of 10, ge=0, le=10')

In [83]:
structured_model = model.with_structured_output(EvaluationSchema)

In [84]:
essay = """Artificial Intelligence (AI) is rapidly transforming the world, and **India is emerging as one of the key players in adopting and developing AI technologies**. With its strong IT industry, large talent pool, and growing digital infrastructure, AI is playing an increasingly important role in shaping India’s economy, governance, education, healthcare, and daily life. 🇮🇳🤖

One of the most significant roles of AI in India is in the **healthcare sector**. AI-powered tools are helping doctors diagnose diseases more accurately and quickly. Technologies such as medical imaging analysis, predictive health monitoring, and virtual health assistants are improving patient care, especially in rural areas where access to specialist doctors is limited. AI is also supporting drug discovery and improving hospital management systems, making healthcare more efficient and affordable.

AI is also transforming **education in India**. With the help of intelligent learning platforms, students can now receive personalized learning experiences based on their abilities and pace. AI-based applications provide interactive content, automated assessments, and real-time feedback to students. These tools are especially useful in bridging the gap between urban and rural education systems and making quality learning accessible to everyone. 📚✨

In the field of **agriculture**, AI is helping farmers increase productivity and reduce risks. Smart farming techniques powered by AI can analyze soil conditions, weather patterns, crop health, and irrigation needs. Farmers can receive timely advice on crop selection, pest control, and fertilizer use through AI-driven mobile applications. This improves crop yield and supports sustainable agricultural practices, which is crucial for a country like India where agriculture employs a large portion of the population.

AI is also playing an important role in **governance and public services**. Government initiatives are increasingly using AI to improve decision-making, enhance transparency, and deliver services efficiently. AI-based systems help in traffic management, smart city development, fraud detection, and digital identity verification. These technologies are making government services faster, more reliable, and citizen-friendly. 🚦🏙️

India’s **economy and employment sector** are also benefiting from AI adoption. AI is creating new job opportunities in areas such as data science, machine learning, robotics, and automation. At the same time, it is improving productivity across industries like banking, manufacturing, retail, and logistics. Startups in India are actively developing innovative AI solutions, contributing to economic growth and global competitiveness.

However, the growth of AI also brings certain challenges. Issues such as data privacy, ethical concerns, job displacement due to automation, and the need for skilled professionals must be addressed carefully. The government and private sector need to work together to ensure responsible AI development and proper training programs for the workforce.

In conclusion, AI is playing a transformative role in India’s development by improving healthcare, education, agriculture, governance, and economic growth. With continued investment in technology, infrastructure, and skill development, India has the potential to become a global leader in artificial intelligence while ensuring inclusive and sustainable progress for its citizens. 🚀
"""

In [85]:
prompt = f"""
Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}
"""
res = structured_model.invoke(prompt)
res

ChatGoogleGenerativeAIError: Error calling model 'gemini-3-flash-preview' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3-flash\nPlease retry in 29.800332789s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-F

In [ ]:
class EssayState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_score: Annotated[list[int], operator.add]
    avg_score: float

In [ ]:
def evaluate_language(state: EssayState) -> EssayState:
    prompt = f"""
    Evaluate the language quality of the following essay
    and provide feedback and score out of 10

    {state['essay']}
    """

    output = structured_model.invoke(prompt)

    return {
        "language_feedback": output["feedback"],
        "individual_score": [output["score"]],
    }

In [ ]:
def evaluate_analysis(state: EssayState) -> EssayState:

    prompt = f"""
    Evaluate depth of analysis of the following essay
    and provide feedback and score out of 10

    {state['essay']}
    """

    output = structured_model.invoke(prompt)

    return {
        "analysis_feedback": output["feedback"],
        "individual_score": [output["score"]],
    }

In [ ]:
def evaluate_thought(state: EssayState) -> EssayState:

    prompt = f"""
    Evaluate clarity of thought of the following essay
    and provide feedback and score out of 10

    {state['essay']}
    """

    output = structured_model.invoke(prompt)

    return {
        "clarity_feedback": output["feedback"],
        "individual_score": [output["score"]],
    }

In [ ]:
def final_evaluation(state: EssayState) -> EssayState:

    prompt = f"""
    Based on following feedback create summarized feedback:

    language feedback:
    {state['language_feedback']}

    analysis feedback:
    {state['analysis_feedback']}

    clarity feedback:
    {state['clarity_feedback']}
    """

    overall_feedback = structured_model.invoke(prompt)

    avg_score = sum(state["individual_score"]) / len(state["individual_score"])

    return {
        "overall_feedback": overall_feedback["feedback"],
        "avg_score": avg_score,
    }

In [ ]:
graph = StateGraph(EssayState)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

In [ ]:
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge('final_evaluation', END)

In [ ]:
workflow = graph.compile()

In [ ]:
intial_state = {
    'essay': essay
}
result = workflow.invoke(intial_state)
print(result)